In [3]:
# -----------------------------
# 1. Importations
# -----------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [4]:
# -----------------------------
# 1. Jeu de phrases
# -----------------------------
sentences = [
    ["i", "am", "happy"],
    ["i", "love", "this", "movie"],
    ["this", "course", "is", "easy"],
    ["i", "am", "sad"],
    ["this", "movie", "is", "bad"],
    ["i", "hate", "this", "work"],
    ["i", "feel", "great"],
    ["this", "class", "is", "fun"],
    ["i", "am", "tired"],
    ["this", "movie", "is", "terrible"]
]

labels = [1,1,1, 0,0,0, 1,1,0,0]  # 1=positif, 0=négatif

max_len = 6
glove_dim = 50  # plus petit pour test rapide

# -----------------------------
# 2. Créer des embeddings "faux GloVe" aléatoires
# -----------------------------
vocab = set(word for sent in sentences for word in sent)
fake_glove = {word: np.random.randn(glove_dim) for word in vocab}

def sentence_to_vectors(sentence, embeddings_index, max_len=max_len, dim=glove_dim):
    vectors = []
    for word in sentence:
        if word in embeddings_index:
            vectors.append(embeddings_index[word])
        else:
            vectors.append(np.zeros(dim))
    while len(vectors) < max_len:
        vectors.append(np.zeros(dim))
    return np.array(vectors[:max_len])

X = np.array([sentence_to_vectors(s, fake_glove) for s in sentences])
y = np.array(labels)

# -----------------------------
# 3. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

batch_size = 2
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1)

# -----------------------------
# 4. Modèle LSTM
# -----------------------------
hidden_size = 32
num_classes = 2

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

model = LSTMModel(glove_dim, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# -----------------------------
# 5. Entraînement
# -----------------------------
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss={total_loss/len(train_loader):.4f}")

# -----------------------------
# 6. Évaluation
# -----------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
print(f"Test Accuracy: {100*correct/total:.2f}%")

# -----------------------------
# 7. Test simple
# -----------------------------
test_sentence = ["i", "am", "happy"]
test_vec = sentence_to_vectors(test_sentence, fake_glove)
test_tensor = torch.tensor(test_vec, dtype=torch.float32).unsqueeze(0)
pred = model(test_tensor)
pred_label = torch.argmax(pred, dim=1).item()
print(f"Phrase: {' '.join(test_sentence)}, Prediction: {pred_label}")


Epoch 10, Loss=0.0004
Epoch 20, Loss=0.0001
Epoch 30, Loss=0.0001
Test Accuracy: 50.00%
Phrase: i am happy, Prediction: 1
